# FoodLens：Kaggle 提示词复述诊断（不训练）

在 Kaggle 中选择 **GPU T4**、开启 **Internet**，并在 **Add Input** 挂载装有 `annotations.zip` 和 `images.zip` 的私有 Dataset。逐格执行：校验代码 → 查找 ZIP → 依赖与 CUDA 预检 → 数据和模型 → 诊断 → 下载报告。没有旧 Adapter 也能完整运行基础实验。

In [ ]:
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
SCRATCH = Path("/kaggle/temp/foodlens_prompt_diagnostics")
BRANCH = "feat/member-c-diet-record"
REPO_URL = "https://github.com/liuxy2006/FoodLens.git"
EXPECTED_CODE_COMMIT = ""  # 必填：将已推送到成员 C 分支的完整 40 位 SHA 粘贴在这里
DATASET_DIR = ""  # 留空自动查找唯一一对 ZIP；也可填 /kaggle/input 下的目录
EXISTING_STAGE25_DIR = ""  # 可选：/kaggle/input 下的旧 schema_step25 目录；默认不使用 Adapter
REVISION = "7e3e67edbbed1bf9888184d9df282b700a323964"

if not INPUT_ROOT.is_dir() or not WORKING.is_dir():
    raise RuntimeError("请在 Kaggle Notebook 中运行；需要 /kaggle/input 和 /kaggle/working。")
if not re.fullmatch(r"[0-9a-fA-F]{40}", EXPECTED_CODE_COMMIT):
    raise ValueError("请填写已经推送到成员 C 分支的完整 40 位 commit SHA。")
EXPECTED_CODE_COMMIT = EXPECTED_CODE_COMMIT.lower()

## 校验 GitHub 代码

只克隆成员 C 分支，实际 HEAD 必须等于填写的完整 SHA。现有目录只校验，不自动拉取、切换分支或覆盖；版本不符时请在新会话运行。代码和数据缓存都放在临时盘。

In [ ]:
REPO = SCRATCH / "FoodLens-member-c"
SCRATCH.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}, check=True,
    )

def git_value(*args):
    return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()

CODE_COMMIT = git_value("rev-parse", "HEAD")
if git_value("branch", "--show-current") != BRANCH or git_value("remote", "get-url", "origin") != REPO_URL:
    raise RuntimeError("检出的仓库或分支不匹配；只允许 FoodLens 成员 C 分支。")
if CODE_COMMIT != EXPECTED_CODE_COMMIT:
    raise RuntimeError(f"代码 SHA 不匹配：实际 {CODE_COMMIT}，预期 {EXPECTED_CODE_COMMIT}。")
if git_value("status", "--porcelain"):
    raise RuntimeError("代码目录不干净，请使用新 Kaggle 会话。")

def run_module(module, *args):
    subprocess.run([sys.executable, "-m", module, *map(str, args)], cwd=REPO, check=True)

sys.path.insert(0, str(REPO))
print(BRANCH, CODE_COMMIT)

In [ ]:
from task2_diet_record.kaggle_setup import mounted_dataset_dir

ARCHIVE_SOURCE = mounted_dataset_dir(INPUT_ROOT, DATASET_DIR)
DATA_ROOT = SCRATCH / "foodlens_data"
ARCHIVE_CACHE = SCRATCH / "foodlens_archives"
MODEL_DIR = SCRATCH / "models" / ("SmolVLM-256M-Instruct-" + REVISION[:8])
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
RUN_ROOT = WORKING / "foodlens_prompt_diagnostics" / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
DIAGNOSTICS = RUN_ROOT / "probe"
os.environ["HF_HOME"] = str(SCRATCH / "huggingface")
os.environ["HF_HUB_CACHE"] = str(SCRATCH / "huggingface/hub")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
(RUN_ROOT / "source.json").write_text(json.dumps({
    "platform": "Kaggle", "python": platform.python_version(), "branch": BRANCH,
    "commit": CODE_COMMIT, "model_revision": REVISION, "archive_source": str(ARCHIVE_SOURCE),
}, indent=2), encoding="utf-8")
print("ZIP 输入:", ARCHIVE_SOURCE)
print("报告目录:", RUN_ROOT)

## CUDA 和依赖预检

复用项目的依赖安装保护：在新进程中做实际 CUDA float32 运算，锁定 Kaggle 已安装的 PyTorch 组件，检查 pip dry-run 清单。若要替换 CUDA 组件，立即拒绝。安装后重检 CUDA 版本、实际运算及 Transformers、PEFT、Accelerate 和 SmolVLM API 的导入；任一步失败都停止后续下载和推理。

In [ ]:
GPU_READY = False
try:
    run_module("task2_diet_record.kaggle_setup", "--cuda-report", RUN_ROOT / "cuda_before.json")
    run_module("task2_diet_record.colab_dependencies", "--output", RUN_ROOT / "dependency_install")
    run_module("task2_diet_record.kaggle_setup", "--cuda-report", RUN_ROOT / "cuda_after.json")
except subprocess.CalledProcessError as exc:
    raise RuntimeError("Kaggle CUDA 或依赖预检失败。查看上方错误和报告；如需重试，请使用新的 GPU 会话。") from exc
GPU_READY = True

## 准备数据和固定版本模型

从只读挂载复制 ZIP 到临时盘并解压，已有一致的缓存会复用。准备逻辑核对全部标注及图片路径；test 仅做文件完整性检查。模型只从 Hugging Face 获取指定 revision 的 SmolVLM-256M-Instruct 与 Processor，记录清单并校验下载。缓存只在会话临时盘，不会进入报告。

In [ ]:
if not globals().get("GPU_READY", False):
    raise RuntimeError("请先通过 CUDA 和依赖预检。")
from task2_diet_record.prepare_colab_data import prepare_data

preparation = prepare_data(ARCHIVE_SOURCE, DATA_ROOT, ARCHIVE_CACHE)
(RUN_ROOT / "data_preparation.json").write_text(json.dumps(preparation, indent=2), encoding="utf-8")
print(json.dumps(preparation, ensure_ascii=False, indent=2))

In [ ]:
MODEL_READY = False
if not globals().get("GPU_READY", False):
    raise RuntimeError("请先通过 CUDA 和依赖预检。")
if not (RUN_ROOT / "data_preparation.json").is_file():
    raise RuntimeError("请先验证数据 ZIP 和图片路径。")
manifest_path = RUN_ROOT / "model_manifest.json"
run_module("task2_diet_record.prepare_model", "--revision", REVISION, "--destination", MODEL_DIR,
           "--manifest", manifest_path, "--download")
if json.loads(manifest_path.read_text(encoding="utf-8"))["revision"] != REVISION:
    raise RuntimeError("下载的模型 revision 与固定版本不一致。")
MODEL_READY = True

## 诊断：16 张训练图审计和 12 次零样本生成

调用原诊断模块，逐图核对监督 token、图像输入、Label Mask、Assistant 起点和 EOS。使用同一原始预训练模型在固定的 2 张 train + 2 张 val 图片上运行 A/B/C，共 12 次，保留完整 token 轨迹和原始文本，不修补预测。

只有设置 `EXISTING_STAGE25_DIR` 才额外检查旧 Adapter，并仅用原提示词 A 再生成 4 次。Colab 断点路径只在临时副本中转换，原文件不变。这里没有训练入口。

In [ ]:
if not globals().get("GPU_READY", False) or not globals().get("MODEL_READY", False):
    raise RuntimeError("请先通过预检并准备固定版本的模型。")
if DIAGNOSTICS.exists():
    raise FileExistsError("该运行已有诊断目录；请从路径设置单元创建新的运行目录。")

arguments = ["--model-path", MODEL_DIR, "--revision", REVISION, "--data-root", DATA_ROOT,
             "--output", DIAGNOSTICS, "--max-new-tokens", 512]
if EXISTING_STAGE25_DIR:
    from task2_diet_record.kaggle_setup import stage25_for_kaggle

    old_copy = stage25_for_kaggle(INPUT_ROOT, EXISTING_STAGE25_DIR, SCRATCH / "adapter_inputs" / RUN_ID)
    arguments += ["--existing-stage25-dir", old_copy]
run_module("task2_diet_record.prompt_diagnostics", *arguments)

In [ ]:
audit = json.loads((DIAGNOSTICS / "supervision_audit_16.json").read_text(encoding="utf-8"))
print("监督检查:", audit["passed_samples"], "/", audit["samples"])
for sample in audit["samples_report"]:
    if not sample["all_checks_passed"]:
        print(sample["dish_id"], sample.get("checks"), sample.get("error"))

for variant in ("A", "B", "C"):
    report = json.loads((DIAGNOSTICS / "zero_shot" / f"{variant}.json").read_text(encoding="utf-8"))
    print("PROMPT", variant, report["prompt_sha256"], report["summary"])
    for prediction in report["predictions"]:
        print(prediction["dish_id"], prediction["prediction"], prediction.get("echo"), prediction.get("error"))

bundle = RUN_ROOT.with_suffix(".zip")
if not bundle.exists():
    shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name)
print("完整报告:", RUN_ROOT)
print("在 Kaggle Output 下载报告压缩包:", bundle)

先看是否 16/16 通过，再查看三组 12 条记录中是否有生成错误。完整报告保存输入前缀、只含新增 token 的文本、终止信息、提示词及哈希、Schema 和复述统计。四张图的结果不能代表泛化表现；C 组照抄示例也不能算营养估计正确。报告可能含课程标注内容，请保持 Notebook 和输出私有。